<a href="https://colab.research.google.com/github/laveshburla12-commits/Supervised-Machine-Learning/blob/main/KASUSHAL_SIR_SLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("grandmaster07/student-exam-performance-dataset-analysis")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# List files in the downloaded dataset directory to find the CSV
dataset_files = os.listdir(path)
print(f"Files in the dataset directory: {dataset_files}")

# Corrected: Assuming the main data file is 'StudentPerformanceFactors.csv'
data_file_name = 'StudentPerformanceFactors.csv'
full_data_path = os.path.join(path, data_file_name)

# Load the dataset
df_students = pd.read_csv(full_data_path)
display(df_students.head())

### Data Preprocessing

I will create a binary target variable `passed_math` based on whether the `math score` is 70 or higher. This converts the problem into a classification task, which is appropriate for reporting 'accuracy'.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- Data Preprocessing (from cell 21338631) ---
# Assuming df_students is already loaded in the kernel state.
# Create a binary target variable: 1 if Exam_Score >= 70, else 0
df_students['passed_math'] = (df_students['Exam_Score'] >= 70).astype(int)

# Define features (X) and target (y)
X = df_students.drop(['Exam_Score', 'passed_math'], axis=1)
y = df_students['passed_math']

# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns

# Create a preprocessor for one-hot encoding categorical features and scaling numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', RobustScaler(), numerical_features) # Apply RobustScaler to numerical features
    ],
    remainder='passthrough' # Ensure any other unlisted columns are passed through
)

# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# --- Model Training and Evaluation (from cell c4d2cb9b) ---
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear'),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    # Create a pipeline with preprocessing and the model
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', model)])

    # Train the model
    pipeline.fit(X_train, y_train)

    # Make predictions
    y_pred = pipeline.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    print(f"{name} Accuracy: {accuracy:.4f}")

print("\n--- Model Accuracies ---")
for name, accuracy in results.items():
    print(f"{name}: {accuracy:.4f}")

# --- Detailed Model Performance Metrics (from cell 4a381fb5) ---
print("\n--- Detailed Model Performance Metrics ---")
for name, model_accuracy in results.items():
    print(f"\nModel: {name}")
    # Re-train the pipeline to get predictions (or if pipelines were stored, use them directly)
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', models[name])])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print(f"  Accuracy:  {model_accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("\n--- Detailed Model Performance Metrics ---")
for name, model_accuracy in results.items():
    print(f"\nModel: {name}")
    # Re-train the pipeline to get predictions (or if pipelines were stored, use them directly)
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', models[name])])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print(f"  Accuracy:  {model_accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")


The output above shows the coefficients for each feature in the Logistic Regression model, along with the intercept. These values are crucial for understanding how each input variable contributes to the probability of a student passing math.

*   **Coefficients:**
    *   Each coefficient indicates the change in the **log-odds** of the target variable (passing math) for a one-unit increase in the corresponding feature, holding all other features constant.
    *   **Positive coefficients** (e.g., `Hours_Studied`, `Parental_Involvement_High`) mean that an increase in that feature is associated with an increase in the likelihood of passing math.
    *   **Negative coefficients** (e.g., `Parental_Involvement_Low`, `Financial_Support_No`) mean that an increase in that feature is associated with a decrease in the likelihood of passing math.
    *   The **magnitude** of the coefficient indicates the strength of the relationship. Larger absolute values suggest a stronger influence on the outcome.

*   **Intercept:**
    *   The intercept represents the log-odds of the student passing math when all predictor variables are zero (or at their baseline level for one-hot encoded categories).

Essentially, the bar plot visually emphasizes which features have the strongest positive or negative impact on a student's probability of passing the math exam. For instance, a long positive bar for 'Hours_Studied' would mean more study hours significantly increase the chances of passing, while a long negative bar for 'Financial_Support_No' would suggest a significant decrease in chances.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create a DataFrame from the results dictionary
model_comparison_df = pd.DataFrame(results.items(), columns=['Model', 'Accuracy'])

# Sort the DataFrame by Accuracy for better visualization
model_comparison_df = model_comparison_df.sort_values(by='Accuracy', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='Accuracy', data=model_comparison_df, palette='viridis', hue='Model', legend=False)
plt.title('Model Accuracy Comparison')
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.ylim(0.8, 1.0) # Set y-axis limits to better highlight differences
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import RobustScaler

# Create a binary target variable: 1 if Exam_Score >= 70, else 0
df_students['passed_math'] = (df_students['Exam_Score'] >= 70).astype(int)

# Define features (X) and target (y)
X = df_students.drop(['Exam_Score', 'passed_math'], axis=1)
y = df_students['passed_math']

# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns

# Create a preprocessor for one-hot encoding categorical features and scaling numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', RobustScaler(), numerical_features) # Apply RobustScaler to numerical features
    ],
    remainder='passthrough' # Ensure any other unlisted columns are passed through
)

# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

### Model Training and Evaluation

Now, I will train and evaluate three classification models: Logistic Regression, Decision Tree Classifier, and Random Forest Classifier.

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear'),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    # Create a pipeline with preprocessing and the model
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', model)])

    # Train the model
    pipeline.fit(X_train, y_train)

    # Make predictions
    y_pred = pipeline.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    print(f"{name} Accuracy: {accuracy:.4f}")

print("\n--- Model Accuracies ---")
for name, accuracy in results.items():
    print(f"{name}: {accuracy:.4f}")

CHOOSING THE LOGISTIC REGRESSION AS BEST COZ OF ITS ACCURACY SO BELOW ARE THE DATA OF DATASETS WHICH IS FOUND BY MODEL AFTER CHOOSING RIGHT MODEL


In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Get the trained Logistic Regression pipeline
logistic_regression_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                               ('classifier', models['Logistic Regression'])])
logistic_regression_pipeline.fit(X_train, y_train)

# Extract the trained Logistic Regression model
log_reg_model = logistic_regression_pipeline.named_steps['classifier']

# Get feature names after one-hot encoding
one_hot_features = logistic_regression_pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = list(one_hot_features) + list(numerical_features)

# Display coefficients
print("Logistic Regression Coefficients:")
for feature, coef in zip(all_feature_names, log_reg_model.coef_[0]):
    print(f"  {feature}: {coef:.4f}")

print(f"\nLogistic Regression Intercept: {log_reg_model.intercept_[0]:.4f}")

This bar plot visually represents the impact of each feature on the log-odds of a student passing math. Longer bars indicate a stronger influence, and the color (or position on the x-axis) shows whether the influence is positive or negative. For example, a high positive coefficient means that feature strongly increases the probability of passing, while a high negative coefficient strongly decreases it.

In [ ]:
import pandas as pd

# Create a sample student's data using a dictionary
# Ensure all original features are present, with appropriate data types
sample_student_data = {
    'Hours_Studied': [25], # Example: student studied 25 hours
    'Attendance': [95],    # Example: 95% attendance
    'Parental_Involvement': ['High'],
    'Access_to_Resources': ['High'],
    'Extracurricular_Activities': ['Yes'],
    'Motivation_Level': ['High'],
    'Internet_Access': ['Yes'],
    'Family_Income': ['Medium'],
    'Teacher_Quality': ['High'],
    'School_Type': ['Public'],
    'Peer_Influence': ['Positive'],
    'Learning_Disabilities': ['No'],
    'Parental_Education_Level': ['Postgraduate'],
    'Distance_from_Home': ['Near'],
    'Gender': ['Female'],
    'Sleep_Hours': [8],    # Example: 8 hours of sleep
    'Previous_Scores': [80], # Example: previous average score of 80
    'Tutoring_Sessions': [1], # Example: attended tutoring sessions
    'Physical_Activity': [5]  # Example: 5 hours of physical activity per week
}

sample_df = pd.DataFrame(sample_student_data)

# Make a prediction using the best-performing Logistic Regression pipeline
# The pipeline already contains the preprocessor and the classifier

# Assuming `logistic_regression_pipeline` is still available from previous execution
# If not, it would need to be re-created by training the model again.
# For robustness, let's ensure the pipeline is available or re-create it
if 'logistic_regression_pipeline' not in locals():
    logistic_regression_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                                   ('classifier', models['Logistic Regression'])])
    logistic_regression_pipeline.fit(X_train, y_train)

prediction = logistic_regression_pipeline.predict(sample_df)
prediction_proba = logistic_regression_pipeline.predict_proba(sample_df)

# Interpret the prediction
if prediction[0] == 1:
    result = 'Pass'
else:
    result = 'Fail'

print(f"Prediction for the sample student: {result}")
print(f"Probability of Passing: {prediction_proba[0][1]:.4f}")
print(f"Probability of Failing: {prediction_proba[0][0]:.4f}")

# Task
Create an interactive prediction widget using `ipywidgets`. This widget will allow users to input values for all features used by the Logistic Regression model (e.g., `Hours_Studied`, `Attendance`, `Parental_Involvement`, etc.) through sliders for numerical features and dropdowns for categorical features. The widget will then display the predicted pass/fail status and associated probabilities based on the user's input.

## Create Interactive Input Widgets

### Subtask:
Generate interactive input widgets (sliders for numerical features and dropdowns for categorical features) using `ipywidgets` based on the features used in the model. Default values will be set for demonstration.


**Reasoning**:
I will import the necessary `ipywidgets` components, then iterate through the features in `X_train` to create `FloatSlider` widgets for numerical features and `Dropdown` widgets for categorical features, storing them in a dictionary.



In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Identify numerical and categorical features from X (used in preprocessor)
# These lists should already be available from the preprocessing step
# If not, they would need to be re-derived:
# categorical_features = X.select_dtypes(include=['object']).columns
# numerical_features = X.select_dtypes(include=['int64', 'float64']).columns

feature_widgets = {}

print("Generating widgets for features...")

# Create widgets for numerical features
for feature in numerical_features:
    min_val = X_train[feature].min()
    max_val = X_train[feature].max()
    # Using a step of 1 for integer-like values, or a smaller step for floats if necessary
    step = 1 if X_train[feature].dtype in ['int64', 'int32'] else (max_val - min_val) / 100.0
    default_val = X_train[feature].mean() # Using mean as default

    slider = widgets.FloatSlider(
        value=default_val,
        min=min_val,
        max=max_val,
        step=step,
        description=feature.replace('_', ' ') + ':',
        orientation='horizontal',
        continuous_update=False, # Update only on release
        readout=True,
        readout_format='.2f',
        layout=widgets.Layout(width='500px')
    )
    feature_widgets[feature] = slider
    # display(slider) # Only display for demonstration, not in final output

# Create widgets for categorical features
for feature in categorical_features:
    options = X_train[feature].unique().tolist()
    # Handle potential NaN values if present in options by removing them
    options = [str(opt) for opt in options if pd.notna(opt)]

    # Sort options for better presentation, if they are not too many
    if len(options) < 20: # Arbitrary threshold for sorting
        options.sort()

    default_val = X_train[feature].mode()[0] # Using mode as default

    dropdown = widgets.Dropdown(
        options=options,
        value=str(default_val), # Ensure value is string if options are strings
        description=feature.replace('_', ' ') + ':',
        disabled=False,
        layout=widgets.Layout(width='500px')
    )
    feature_widgets[feature] = dropdown
    # display(dropdown) # Only display for demonstration, not in final output

print(f"Created {len(feature_widgets)} widgets.")
print("Example widget (Hours_Studied):", feature_widgets['Hours_Studied'])
print("Example widget (Parental_Involvement):", feature_widgets['Parental_Involvement'])

**Reasoning**:
Now that the individual feature widgets are created, the next step is to define a prediction function that will take the values from these widgets, transform them into a DataFrame suitable for the model, and then make a prediction using the trained Logistic Regression pipeline. This function will be called whenever the widget values change.



In [ ]:
def predict_and_display(**kwargs):
    # Create a DataFrame from the current widget values
    current_data = {feature: [value] for feature, value in kwargs.items()}
    sample_df_live = pd.DataFrame(current_data)

    # Ensure the order of columns in sample_df_live matches X_train
    sample_df_live = sample_df_live[X_train.columns]

    # Make a prediction using the best-performing Logistic Regression pipeline
    prediction = logistic_regression_pipeline.predict(sample_df_live)
    prediction_proba = logistic_regression_pipeline.predict_proba(sample_df_live)

    # Interpret the prediction
    if prediction[0] == 1:
        result = 'Pass'
    else:
        result = 'Fail'

    # Display the results
    with output_widget:
        output_widget.clear_output()
        print(f"Prediction for the student: {result}")
        print(f"Probability of Passing: {prediction_proba[0][1]:.4f}")
        print(f"Probability of Failing: {prediction_proba[0][0]:.4f}")

# Create an Output widget to display results dynamically
output_widget = widgets.Output()
display(output_widget)

**Reasoning**:
To complete the interactive widget, I need to arrange all the created feature widgets, link them to the prediction function, and display the interactive interface to the user.



## Create Predict Button

### Subtask:
Create an `ipywidgets.Button` instance with the label 'Get Prediction'. This button will be used to explicitly trigger a prediction.


**Reasoning**:
To fulfill the subtask, I will create an `ipywidgets.Button` instance with the specified label 'Get Prediction'.



In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Create a button to trigger prediction
prediction_button = widgets.Button(description="Get Prediction")

print("Prediction button created.")

## Define Button Click Handler

### Subtask:
Implement a Python function that will be executed when the 'Get Prediction' button is clicked. This function will collect the current values from all the `feature_widgets`, assemble them into a dictionary, then pass these values to the `predict_and_display` function to perform and show the prediction in the `output_widget`.


**Reasoning**:
I will define the `on_button_click` function as specified, which will collect values from all interactive widgets and then invoke the `predict_and_display` function to show the prediction.



In [ ]:
def on_button_click(button):
    current_input_values = {}
    for feature, widget in feature_widgets.items():
        current_input_values[feature] = widget.value

    # Call the prediction function with the collected values
    predict_and_display(**current_input_values)

**Reasoning**:
Now that the `on_button_click` function is defined and the `prediction_button` is created, I need to link the function to the button's click event. Then, I will display all the input widgets, the button, and the output area to create the complete interactive prediction interface.



In [ ]:
from ipywidgets import VBox, HBox

# Link the button to the prediction function
prediction_button.on_click(on_button_click)

# Group widgets for display
input_widgets_layout = VBox(list(feature_widgets.values()))

# Display the input widgets, the button, and the output
print("Adjust the student's characteristics below and click 'Get Prediction':")
display(input_widgets_layout, prediction_button, output_widget)

## Summary:

### Data Analysis Key Findings

*   An `ipywidgets.Button` instance, labeled "Get Prediction", was successfully created to serve as the explicit trigger for predictions.
*   A Python function, `on_button_click`, was implemented to handle the button's click event. This function gathers current values from all `feature_widgets`, compiles them into a dictionary, and then invokes the `predict_and_display` function using these values.
*   The newly created `prediction_button` was successfully linked to the `on_button_click` function.
*   The complete interactive prediction interface, comprising input widgets, the "Get Prediction" button, and an output display area, was successfully arranged and displayed.

### Insights or Next Steps

*   The implementation of an explicit "Get Prediction" button enhances user control by allowing predictions to be triggered only when desired, preventing continuous and potentially resource-intensive predictions as input values change.
*   This setup provides a clear workflow: modify inputs, then explicitly request a prediction, which can improve the user experience, especially for models with longer inference times or when users want to experiment with different input combinations before requesting a final prediction.


In [ ]:
1.

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

# Step 1: Load the dataset
iris = load_iris()
X = iris.data
y = iris.target
print("Abhishek Kumar Singh,25FMSCIT001")

# --- SUPERVISED LEARNING (Classification) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=50)
sup_model = LogisticRegression(max_iter=500)
sup_model.fit(X_train, y_train)
y_pred = sup_model.predict(X_test)

print("\nSupervised Learning Accuracy:", accuracy_score(y_test, y_pred))

# --- UNSUPERVISED LEARNING (Clustering) ---
unsup_model = KMeans(n_clusters=3, random_state=50)
clusters = unsup_model.fit_predict(X)

# Step 3: Visualizing the Clusters
plt.figure(figsize=(8,6))
plt.scatter(X[clusters == 0, 0], X[clusters == 0, 1], s = 100, c = 'red', label = 'Cluster 1')
plt.scatter(X[clusters == 1, 0], X[clusters == 1, 1], s = 100, c = 'blue', label = 'Cluster 2')
plt.scatter(X[clusters == 2, 0], X[clusters == 2, 1], s = 100, c = 'green', label = 'Cluster 3')

# Plotting the centroids
plt.scatter(unsup_model.cluster_centers_[:, 0], unsup_model.cluster_centers_[:, 1],
            s = 300, c = 'yellow', label = 'Centroids', marker='*')

plt.title('Abhishek Kumar Singh,25FMSCIT001, Unsupervised Learning: K-Means Clustering')
plt.xlabel('Sepal Length')
plt.ylabel('Sepal Width')
plt.legend()
plt.grid(True)
plt.show()

print("\nUnsupervised Learning Labels assigned:\n", clusters)


In [ ]:
2.

#Binary Classification

import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#step 2: load dataset
iris = load_iris()
X = iris.data
y = iris.target
print("Abhishek Kumar Singh,25FMSCIT001")
print(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=50)
model = LogisticRegression(max_iter=210)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Score: ", accuracy_score(y_test, y_pred))



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

print("Abhishek Kumar Singh,25FMSCIT001")

# Generate synthetic data for demonstration
np.random.seed(42)
x = np.random.rand(100, 1) * 10  # 100 samples between 0 and 10
y = 2 * x**2 - 5 * x + 3 + np.random.randn(100, 1) * 5 # Quadratic relationship with noise

# Create a dummy DataFrame for head display, as it was in the original code
df = pd.DataFrame({'X': x.flatten(), 'y': y.flatten()})
print(df.head())

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.4, random_state = 35)
train_errors = []
test_errors = []

degrees = list(range(1, 11))

for d in degrees:
    poly = PolynomialFeatures(degree=d)
    x_train_poly = poly.fit_transform(x_train)
    x_test_poly = poly.transform(x_test)
    model = LinearRegression()
    model.fit(x_train_poly, y_train)
    y_train_pred = model.predict(x_train_poly)
    y_test_pred = model.predict(x_test_poly)
    train_errors.append(mean_squared_error(y_train, y_train_pred))
    test_errors.append(mean_squared_error(y_test, y_test_pred))

plt.figure(figsize=(10,6))
plt.plot(degrees, train_errors, marker='o', label='Training Error')
plt.plot(degrees, test_errors, marker='x', label='Testing Error')
plt.xlabel('Polynomial Degree')
plt.ylabel('Mean Squared Error')
plt.title('Abhishek Kumar Singh,25FMSCIT001, Training versus Testing')
plt.legend()
plt.show()

In [ ]:
4.
import numpy as np
import matplotlib.pyplot as plt


print("Abhishek Kumar Singh,25FMSCIT001")
# x = study hours, independent variable
# y = exam score, dependent variable
x = np.array([1,2,3,4,5,6,7])
y = np.array([2,4,5,4,6,7,8]) # Corrected: Added one value to match x's length

n = len(x)
sum_x = np.sum(x)
sum_y = np.sum(y)

sum_xy = np.sum(x*y)
sum_x2 = np.sum((x*x))

#slope/ coefficient
m = (n*sum_xy-sum_x*sum_y)/(n*sum_x2-sum_x**2)
#intercept
b = (sum_y-m*sum_x)/n

print("Slope: ", m)
print("Intercept: ", b)

y_pred = m*x+b

#visualization
plt.scatter(x, y, color='blue', label='actual data')
plt.plot(x, y_pred, color='red', label='Best Fit Line')
plt.xlabel('Study Hours')
plt.ylabel('Exam Score')
plt.title('Abhishek Kumar Singh,25FMSCIT001, Linear Regression using Least Square')
plt.legend()
plt.show()

#Code 2
from sklearn.linear_model import LinearRegression
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1,2,3,4,5,6,7]).reshape(-1,1)
y = np.array([2,4,5,4,6,7,8]) # Corrected: Added one value to match x's length

model = LinearRegression()
model.fit(x,y)
print("Slope: ", model.coef_[0])
print("Intercept: ", model.intercept_)

In [ ]:
5.

import numpy as numpy
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier

print("Abhishek Kumar Singh,25FMSCIT001")

iris = load_iris()
X = iris.data
Y = iris.target

df = pd.DataFrame(X, columns=iris.feature_names)
df['species'] = iris.target

X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.4, random_state=50)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, Y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(Y_test, y_pred)
precision = precision_score(Y_test, y_pred, average='weighted')
recall = recall_score(Y_test, y_pred, average='weighted')
f1 = f1_score(Y_test, y_pred, average='weighted')

print("Accuracy Score: ", accuracy)
print("Precision Score: ", precision)
print("Recall Score: ", recall)
print("F1 Score: ", f1)
print('Classificaton Report: ')
print(classification_report(Y_test, y_pred, target_names = iris.target_names))













In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import kagglehub
import os

print("Abhishek Kumar Singh,25FMSCIT001")

# Download the dataset
path = kagglehub.dataset_download('vjchoudhary7/customer-segmentation-tutorial-in-python')

# List files in the downloaded dataset directory to find the CSV
dataset_files = os.listdir(path)
print(f"Files in the dataset directory: {dataset_files}")

# Assuming the main data file is 'Mall_Customers.csv'
data_file_name = 'Mall_Customers.csv'
full_data_path = os.path.join(path, data_file_name)

df = pd.read_csv(full_data_path)
print(df)

x = df[['Annual Income (k$)', 'Spending Score (1-100)']].values
wcss = []
for i in range(1,11):
    kmeans = KMeans(n_clusters=i, init= 'k-means++', random_state=60, n_init=10)
    kmeans.fit(x)
    wcss.append(kmeans.inertia_)

plt.plot(range(1,11),wcss, marker='o')
plt.title('Abhishek Kumar Singh,25FMSCIT001, Elbow Method')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.grid(True)
plt.show()

kmeans = KMeans(n_clusters=10, init='k-means++', random_state=60, n_init=15)
y_kmeans = kmeans.fit_predict(x)
plt.figure(figsize=(10,6))

plt.scatter(x[y_kmeans==0, 0], x[y_kmeans==0, 1], s=80, c='red', label='Cluster 1')
plt.scatter(x[y_kmeans==1, 0], x[y_kmeans==1, 1], s=80, c='blue', label='Cluster 2')
plt.scatter(x[y_kmeans==2, 0], x[y_kmeans==2, 1], s=80, c='green', label='Cluster 3')
plt.scatter(x[y_kmeans==3, 0], x[y_kmeans==3, 1], s=80, c='cyan', label='Cluster 4')
plt.scatter(x[y_kmeans==4, 0], x[y_kmeans==4, 1], s=80, c='pink', label='Cluster 5')

plt.scatter(kmeans.cluster_centers_[:,0], kmeans.cluster_centers_[:,1], s=300, c ='yellow', marker='x', label='Centroids')
plt.title('Abhishek Kumar Singh,25FMSCIT001, Customer Segmentation')
plt.xlabel('Annual Income')
plt.ylabel('Spending Score')
plt.legend()
plt.show()

In [ ]:
7.
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

print("Abhishek Kumar Singh,25FMSCIT001")

# load dataset
iris = load_iris()
X = iris.data
y = iris.target
print(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = DecisionTreeClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Score: ", accuracy_score(y_test, y_pred))
print("Classification Report\n :", classification_report(y_test, y_pred))





In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, completeness_score
import kagglehub
import os

print("Abhishek Kumar Singh,25FMSCIT001")

# Download the dataset
path = kagglehub.dataset_download('vjchoudhary7/customer-segmentation-tutorial-in-python')

# List files in the downloaded dataset directory to find the CSV
dataset_files = os.listdir(path)
print(f"Files in the dataset directory: {dataset_files}")

# Assuming the main data file is 'Mall_Customers.csv'
data_file_name = 'Mall_Customers.csv'
full_data_path = os.path.join(path, data_file_name)

df = pd.read_csv(full_data_path)
x = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']]

scaler = StandardScaler()
x_scale = scaler.fit_transform(x)

hc = AgglomerativeClustering(n_clusters=3, linkage='ward')
cluster_labels = hc.fit_predict(x_scale)
df['Cluster'] = cluster_labels

sil_score = silhouette_score(x_scale, df['Cluster'])
comp_score = completeness_score(df['Cluster'], cluster_labels)

print(df.head())
print("Silhouette Score: ", sil_score)
print("Completeness Score: ", comp_score)